# Citizen Grievance AI — Model Comparison & Stratified Group Evaluation

This notebook loads `data/synthetic_complaints.csv` (5,500 multilingual civic complaints across 6 departments and 15 Hyderabad wards), performs a **template-group-aware split (`StratifiedGroupKFold`)** so paraphrases of the same template never leak between train and validation/test sets, and benchmarks:
- `MultinomialNB`
- `LogisticRegression`
- `LinearSVC` (Calibrated)
- `RandomForestClassifier`

It also runs a **feature ablation study** (word n-grams vs char n-grams vs combined) and **per-language slice evaluation** (`en`, `te`, `hi`, `mixed`).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.trainer import load_training_dataset, _stratified_group_holdout, _evaluate_task

df = load_training_dataset()
print(f'Dataset rows: {len(df)} | Unique template groups: {df["template_group"].nunique()}')
df[['department', 'priority', 'language']].describe(include='all')

In [ ]:
train_df, test_df = _stratified_group_holdout(df, target_col='department', n_splits=5)
overlap = set(train_df['template_group']) & set(test_df['template_group'])
print(f'Train rows: {len(train_df)} | Test rows (unseen templates): {len(test_df)} | Template leakage: {len(overlap)}')
assert len(overlap) == 0

In [ ]:
dept_eval = _evaluate_task(train_df, test_df, target_col='department', cv_splits=5)
pd.DataFrame(dept_eval['model_comparison'])

In [ ]:
prio_eval = _evaluate_task(train_df, test_df, target_col='priority', cv_splits=5)
pd.DataFrame(prio_eval['model_comparison'])

In [ ]:
print('--- Department Feature Ablation ---')
display(pd.DataFrame(dept_eval['ablation']))
print('--- Department Per-Language Evaluation ---')
display(pd.DataFrame.from_dict(dept_eval['language_slices'], orient='index'))